# **Complete Working Process of Recurrent Neural Networks (RNN)**



This document describes the complete mathematical and structural lifecycle of a Recurrent Neural Network (RNN) during a single forward and backward pass.


## 1. Input Representation (The 3D Tensor)

An RNN expects input data to be formatted as a 3D tensor of shape:

$$\text{Input Shape} = (\text{Batch Size}, \text{Timesteps}, \text{Input Features})$$

For example, when feeding a batch of sentences:
*   **Batch Size ($B$)**: The number of sentences processed together (e.g., 32).
*   **Timesteps ($T$)**: The number of words (sequence length) per sentence (e.g., 5).
*   **Input Features ($D$)**: The size of the vector representing each word/step (e.g., 128).


## 2. Temporal Unrolling

An RNN contains a single cell with recurrent connections. When processing a sequence of length $T$, we visualize the loop by **unrolling** it across time:

```mermaid
graph LR
    subgraph Step 1
        x1["Input x_1"] --> cell1["RNN Cell (h_1)"]
        h0["Init h_0 (Zeros)"] --> cell1
        cell1 --> y1["Output y_1"]
    end
    subgraph Step 2
        x2["Input x_2"] --> cell2["RNN Cell (h_2)"]
        cell1 -->|Memory Transfer| cell2
        cell2 --> y2["Output y_2"]
    end
    subgraph Step 3
        x3["Input x_3"] --> cell3["RNN Cell (h_3)"]
        cell2 -->|Memory Transfer| cell3
        cell3 --> y3["Output y_3"]
    end

    style cell1 fill:#e1f5fe,stroke:#01579b,stroke-width:2px
    style cell2 fill:#e1f5fe,stroke:#01579b,stroke-width:2px
    style cell3 fill:#e1f5fe,stroke:#01579b,stroke-width:2px
```

At every step, the cell uses the **exact same weight parameters** ($W_{xh}$, $W_{hh}$, and $W_{hy}$).


## 3. The Forward Propagation Pass (Step-by-Step)

Suppose we have an input sequence of length $T$: $\{x_1, x_2, \dots, x_T\}$.

### Step A: Initialize the Hidden State
Before reading the first input, the hidden state (memory) is initialized to zero:
$$h_0 = \vec{0}$$

### Step B: The Recurrent Loop
For each time step $t$ from $1$ to $T$, the RNN performs the following calculations:

#### 1. Combine Input and Previous Memory
The cell projects the current input vector $x_t$ and the previous hidden state $h_{t-1}$ using its weight matrices:
$$\text{Input Projection} = W_{xh} x_t$$
$$\text{Memory Projection} = W_{hh} h_{t-1}$$

#### 2. Update the Hidden State
The projections are summed with a bias vector $b_h$, and passed through a non-linear activation function (usually hyperbolic tangent, $\tanh$):
$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

$$Total Sum = (Input Signal) + (W11 * h1 prev) + (W12 * h2 prev) + (W13 * h3 prev) + b1$$


*This updated $h_t$ is now the active memory state, representing everything the model has learned from step $1$ to step $t$.*

**IMP**<br>
- $W_{xh}$ (Input Weights): Through thousands of examples, the model learns which features in your input data are statistically relevant to the goal. If a certain input feature consistently leads to an error, the model pushes the weight for that feature toward zero (making it "forget" that input).
- $W_{hh}$ (Recurrent Weights): This is the "memory" weight. The model learns how to structure these weights so that important information from the past is preserved in the hidden state $h_t$ while unimportant noise is discarded.



#### 3. Compute the Output
To make a prediction at the current step $t$, the hidden state is projected to the output space and mapped through an activation function (like `softmax` for classification):
$$y_t = \text{softmax}(W_{hy} h_t + b_y)$$


## 4. Loss Calculation

Once the model completes the forward pass through all $T$ timesteps, the loss is calculated. 
*   **Many-to-One (e.g., Sentiment analysis)**: The loss is computed using only the final step's prediction $y_T$ compared against the true target label $Y$:
    $$\mathcal{L} = \text{Loss}(y_T, Y)$$
*   **Many-to-Many (e.g., Translation)**: The loss is summed over all time steps:
    $$\mathcal{L} = \sum_{t=1}^{T} \text{Loss}(y_t, Y_t)$$


## 5. Backpropagation Through Time (BPTT)

To update the weights ($W_{xh}, W_{hh}, W_{hy}$), the network uses a modified version of backpropagation called **Backpropagation Through Time (BPTT)**:

1.  **Backward Flow**: The loss gradient is calculated at the final step $T$.
2.  **Temporal Chain Rule**: The gradients flow backward chronologically from step $T$ to step $1$.
3.  **Weight Updates**: Because the weight matrices are shared across all timesteps, the gradient for each weight matrix is the sum of the gradients calculated at each individual time step:
    $$\frac{\partial \mathcal{L}}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial \mathcal{L}_t}{\partial W_{hh}}$$

### The Vanishing Gradient Problem
During BPTT, calculating the gradient at step $1$ requires multiplying the recurrent weights ($W_{hh}$) repeatedly:
$$\frac{\partial h_t}{\partial h_1} = \prod_{k=2}^{t} \frac{\partial h_k}{\partial h_{k-1}}$$
If the values of $W_{hh}$ are slightly less than $1.0$, multiplying them repeatedly over 100 timesteps causes the gradient to shrink exponentially to zero. As a result, the model's early weights stop updating, and it "forgets" long-term dependencies.